In [27]:
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F
from credit_risk.utils.config import read_config,create_path
from credit_risk.utils.spark import create_spark_session

In [28]:
project_path = Path.cwd()

In [29]:
import os 
os.chdir(project_path)

In [30]:
config = read_config(project_path)

In [31]:
spark = create_spark_session(config)

In [32]:
macro_path = create_path(
            config["catalog"]["base"],
            config["catalog"],
            "raw_macro_data",
        )

macro = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("dateFormat", "yyyy-MM-dd")
    .csv(str(macro_path))
)

In [33]:
macro.printSchema()
macro.show(10, truncate=False)

root
 |-- period: date (nullable = true)
 |-- unemployment_rate: double (nullable = true)
 |-- mortgage_rate_30y: double (nullable = true)
 |-- hpi_purchase_only: double (nullable = true)
 |-- fed_funds_rate: double (nullable = true)

+----------+-----------------+-----------------+------------------+--------------+
|period    |unemployment_rate|mortgage_rate_30y|hpi_purchase_only |fed_funds_rate|
+----------+-----------------+-----------------+------------------+--------------+
|2014-01-01|6.6              |4.432            |198.7             |0.07          |
|2014-02-01|6.7              |4.3025           |199.68            |0.07          |
|2014-03-01|6.7              |4.3425           |202.09            |0.08          |
|2014-04-01|6.2              |4.3375           |204.31            |0.09          |
|2014-05-01|6.3              |4.192            |206.67            |0.09          |
|2014-06-01|6.1              |4.1625           |209.05            |0.1           |
|2014-07-01|6.2   

In [34]:
MASTER_PATH = "data/03_processed/behavioral/freddie_mac/2015/model-input.parquet"

master = spark.read.parquet(MASTER_PATH)

print("Master rows:", master.count())
master.printSchema()
master.select("period", "loan_id").show(10, truncate=False)

Master rows: 5535631
root
 |-- loan_id: string (nullable = true)
 |-- observation_age: integer (nullable = true)
 |-- period: long (nullable = true)
 |-- current_actual_upb: double (nullable = true)
 |-- current_interest_rate: double (nullable = true)
 |-- loan_age: long (nullable = true)
 |-- remaining_months_to_legal_maturity: long (nullable = true)
 |-- estimated_ltv: long (nullable = true)
 |-- current_loan_delinquency_status: string (nullable = true)
 |-- ddlpi: long (nullable = true)
 |-- modification_flag: string (nullable = true)
 |-- current_non_interest_bearing_upb: double (nullable = true)
 |-- current_interest_bearing_upb: double (nullable = true)
 |-- interest_rate_step_indicator: string (nullable = true)
 |-- payment_deferral_flag: string (nullable = true)
 |-- delinquency_due_to_disaster: string (nullable = true)
 |-- borrower_assistance_plan: string (nullable = true)
 |-- mi_cancellation_indicator: string (nullable = true)
 |-- servicer_name: string (nullable = true)
 |

In [35]:
# Macro currently has:
# period = 2014-01-01, 2014-02-01, ...

macro_spark = macro.withColumn(
    "period",
    (
        (F.year("period") - F.lit(1970)) * 12
        + (F.month("period") - F.lit(1))
    ).cast("long")
)

In [36]:
macro_spark.orderBy("period").show(10)

+------+-----------------+-----------------+------------------+--------------+
|period|unemployment_rate|mortgage_rate_30y| hpi_purchase_only|fed_funds_rate|
+------+-----------------+-----------------+------------------+--------------+
|   528|              6.6|            4.432|             198.7|          0.07|
|   529|              6.7|           4.3025|            199.68|          0.07|
|   530|              6.7|           4.3425|            202.09|          0.08|
|   531|              6.2|           4.3375|            204.31|          0.09|
|   532|              6.3|            4.192|            206.67|          0.09|
|   533|              6.1|           4.1625|            209.05|           0.1|
|   534|              6.2|             4.13|209.77000000000004|          0.09|
|   535|              6.1|            4.115|            209.37|          0.09|
|   536|              5.9|           4.1625|            208.45|          0.09|
|   537|              5.7|            4.036|        

In [22]:
master_macro = master.join(
    macro_spark,
    on="period",
    how="left"
)

In [37]:
master.select(
    "loan_id",
    "period",
    "unemployment_rate",
    "mortgage_rate_30y",
    "hpi_purchase_only",
    "fed_funds_rate"
).orderBy("period").show(30, truncate=False)

+------------+------+-----------------+------------------+------------------+--------------+
|loan_id     |period|unemployment_rate|mortgage_rate_30y |hpi_purchase_only |fed_funds_rate|
+------------+------+-----------------+------------------+------------------+--------------+
|F15Q10003579|543   |5.4              |3.6719999999999997|214.61000000000004|0.12          |
|F15Q10091397|543   |5.4              |3.6719999999999997|214.61000000000004|0.12          |
|F15Q10099669|543   |5.4              |3.6719999999999997|214.61000000000004|0.12          |
|F15Q10004061|543   |5.4              |3.6719999999999997|214.61000000000004|0.12          |
|F15Q10005401|543   |5.4              |3.6719999999999997|214.61000000000004|0.12          |
|F15Q10034822|543   |5.4              |3.6719999999999997|214.61000000000004|0.12          |
|F15Q10024270|543   |5.4              |3.6719999999999997|214.61000000000004|0.12          |
|F15Q10004271|543   |5.4              |3.6719999999999997|214.61000000

In [38]:
master.groupBy("period").agg(
    F.count("*").alias("master_rows"),
    F.first("unemployment_rate").alias("unemployment_rate"),
    F.first("mortgage_rate_30y").alias("mortgage_rate_30y"),
    F.first("hpi_purchase_only").alias("hpi_purchase_only"),
    F.first("fed_funds_rate").alias("fed_funds_rate")
).orderBy("period").show(20, truncate=False)

+------+-----------+-----------------+------------------+------------------+--------------+
|period|master_rows|unemployment_rate|mortgage_rate_30y |hpi_purchase_only |fed_funds_rate|
+------+-----------+-----------------+------------------+------------------+--------------+
|543   |941        |5.4              |3.6719999999999997|214.61000000000004|0.12          |
|544   |88022      |5.6              |3.84              |217.74            |0.12          |
|545   |114671     |5.3              |3.9825            |219.85            |0.13          |
|546   |149068     |5.2              |4.046             |220.6             |0.13          |
|547   |224404     |5.1              |3.9050000000000002|219.92            |0.14          |
|548   |244317     |5.0              |3.89              |219.74            |0.14          |
|549   |276169     |5.0              |3.7960000000000003|219.71999999999997|0.12          |
|550   |341919     |5.1              |3.9425            |219.98            |0.12

In [25]:
master_macro.select(
    F.count("*").alias("total_rows"),
    F.sum(
        F.col("unemployment_rate").isNull().cast("int")
    ).alias("missing_macro_rows")
).show()

+----------+------------------+
|total_rows|missing_macro_rows|
+----------+------------------+
|   5535631|                 0|
+----------+------------------+



In [26]:
master_macro.filter(
    F.col("unemployment_rate").isNull()
).select("period").distinct().orderBy("period").show()

+------+
|period|
+------+
+------+

